In [ ]:
"""
HD-BET: Brain Extraction Tool for MRI Scans
Isensee F, Schell M, Tursunova I, Brugnara G, Bonekamp D, Neuberger U, Wick A,
Schlemmer HP, Heiland S, Wick W, Bendszus M, Maier-Hein KH, Kickingereder P.
Automated brain extraction of multi-sequence MRI using artificial neural
networks. Hum Brain Mapp. 2019; 1-13. https://doi.org/10.1002/hbm.24750
"""

In [ ]:
import nibabel as nib
import numpy as np

def normalise_nii(input_nii_path, output_path):
    """Perform Z-score intensity normalisation only on non-zero voxels."""
    try:
        nii_img = nib.load(input_nii_path)
        image = nii_img.get_fdata()

        mask_brain = image > 0
        if mask_brain.any():
            mean = image[mask_brain].mean()
            std = image[mask_brain].std()
            normalised_image = np.zeros_like(image, dtype=np.float32)
            normalised_image[mask_brain] = (image[mask_brain] - mean) / (std + 1e-8)
        else:
            normalised_image = image.astype(np.float32)

        normalised_nii = nib.Nifti1Image(normalised_image, nii_img.affine, nii_img.header)
        nib.save(normalised_nii, output_path)
        return True
    except Exception as e:
        print(f"Failed to normalise: {input_nii_path}. Got error: {e}")
        return False

In [ ]:
from PIL import Image

def save_png_slices(arr, out_dir, stem):
    """Extract 5 quartile slices from a 3D normalised array and save as uint8 PNGs."""
    os.makedirs(out_dir, exist_ok=True)
    depth = arr.shape[2]
    quartiles = {
        'q25': int(depth * 0.25),
        'q38': int(depth * 0.38),
        'q50': int(depth * 0.50),
        'q62': int(depth * 0.62),
        'q75': int(depth * 0.75),
    }
    for qname, z in quartiles.items():
        sl = arr[:, :, z].T  # transpose to match display convention
        png = np.zeros(sl.shape, dtype=np.uint8)
        brain = sl != 0  # background is zero after skull stripping
        if brain.any():
            brain_vals = sl[brain]
            lo, hi = brain_vals.min(), brain_vals.max()
            png[brain] = ((brain_vals - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)
        Image.fromarray(png).save(os.path.join(out_dir, f"{stem}_{qname}.png"))

In [ ]:
import os
import gc
import shutil

from pathlib import Path
import ants
from HD_BET.run import run_hd_bet
from tqdm.auto import tqdm


def process_single_scan(raw_path, mask_file, png_out_dir, stem, fixed_template):
    try:
        fixed_arr = fixed_template.numpy()
        if fixed_arr.max() == 0:
            print("Fixed_template is completely black. Check template path")
            return False

        # Load raw scan
        raw_img = ants.image_read(str(raw_path)).clone("float")
        
        # Sync mask geometry
        temp_mask_img = ants.image_read(str(mask_file))
        mask_arr = (temp_mask_img.numpy() > 0).astype(np.float32)
        mask_img = raw_img.new_image_like(mask_arr)
        del temp_mask_img

        # Extract brain and N4 correct
        raw_arr = raw_img.numpy()
        raw_arr = np.nan_to_num(raw_arr, nan=0.0, posinf=0.0, neginf=0.0)
        stripped_arr = raw_arr * mask_arr
        stripped_img = raw_img.new_image_like(stripped_arr)
        
        img_n4 = ants.n4_bias_field_correction(stripped_img, mask=mask_img)

        # Scale both images strictly to [0, 1] for the optimizer to calculate 
        # the transform matrix without throwing NaN gradients.
        fixed_reg = fixed_template.new_image_like(
            (fixed_arr - fixed_arr.min()) / (fixed_arr.max() - fixed_arr.min() + 1e-8)
        )
        
        n4_arr = img_n4.numpy()
        img_n4_reg = img_n4.new_image_like(
            (n4_arr - n4_arr.min()) / (n4_arr.max() - n4_arr.min() + 1e-8)
        )

        # Center of mass teleportation
        com_fixed = ants.get_center_of_mass(fixed_reg)
        com_moving = ants.get_center_of_mass(mask_img) 
        
        shift = tuple(f - m for f, m in zip(com_fixed, com_moving))
        new_origin = tuple(o + s for o, s in zip(img_n4.origin, shift))
        
        # Apply origin shift to all moving images
        img_n4.set_origin(new_origin)
        img_n4_reg.set_origin(new_origin)
        mask_img.set_origin(new_origin)

        # Registration: Using Affine and the [0, 1] scaled images
        reg = ants.registration(
            fixed=fixed_reg, 
            moving=img_n4_reg, 
            type_of_transform='Affine',
            verbose=False
        )
        
        # Apply the successful transform to the original non-scaled image
        warped_img_n4 = ants.apply_transforms(
            fixed=fixed_template, 
            moving=img_n4, 
            transformlist=reg['fwdtransforms'], 
            interpolator='linear'
        )
        
        if warped_img_n4.numpy().max() == 0:
            print(f"Registration failed (No overlap/NaN) for {raw_path.name}")
            return False
        
        # Warp the mask using identical spatial transforms
        warped_mask = ants.apply_transforms(
            fixed=fixed_template, moving=mask_img, 
            transformlist=reg['fwdtransforms'], interpolator='nearestNeighbor'
        )
        
        # Re-mask the registered native image cleanly
        clean_arr = warped_img_n4.numpy() * (warped_mask.numpy() > 0)

        # Z-score normalise on brain voxels
        mask_brain = clean_arr > 0
        if mask_brain.any():
            mean = clean_arr[mask_brain].mean()
            std  = clean_arr[mask_brain].std()
            normalised_arr = np.zeros_like(clean_arr, dtype=np.float32)
            normalised_arr[mask_brain] = (clean_arr[mask_brain] - mean) / (std + 1e-8)
        else:
            normalised_arr = clean_arr.astype(np.float32)

        save_png_slices(normalised_arr, png_out_dir, stem)
        return True
            
    except Exception as e:
        print(f"Error for {raw_path.name}: {e}")
        return False



def master_brats_pipeline(input_dir: str, 
                          output_dir: str, 
                          template_path: str, 
                          use_gpu: bool = True):
    
    input_dir, output_dir = Path(input_dir).resolve(), Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    temp_dir = output_dir / "_temp_processing"
    
    # Get all .nii and .nii.gz files in the input directory, excluding any that are already in the output directory
    nii_files = [f for f in input_dir.rglob('*') if f.is_file() and f.name.endswith(('.nii', '.nii.gz')) and output_dir not in f.parents]

    if not nii_files:
        print("No raw files found!")
        return

    files_to_process = []
    for file_path in nii_files:
        rel_path = file_path.relative_to(input_dir)  # Maintain relative subdirectory structure in output

        stem = rel_path.name
        if stem.endswith('.nii.gz'):
            stem = stem[:-7]
        elif stem.endswith('.nii'):
            stem = stem[:-4]

        png_out_dir = output_dir / rel_path.parent / stem

        temp_rel_path = rel_path.with_name(rel_path.name + '.gz') if rel_path.name.endswith('.nii') else rel_path
        temp_hdbet_path = temp_dir / temp_rel_path
            
        if not png_out_dir.exists():
            files_to_process.append({
                "raw_path": file_path,
                "temp_hdbet": temp_hdbet_path,
                "mask_file": temp_hdbet_path.with_name(temp_hdbet_path.name.replace(".nii.gz", "_mask.nii.gz")),
                "png_out_dir": png_out_dir,
                "stem": stem,
            })

    if not files_to_process:
        print("Dataset is already fully processed!")
        return
    
    # --- Native HD-BET Mask Generation (GPU) for skull masking ---
    paths_for_hdbet_in = []
    paths_for_hdbet_out = []
    
    for item in files_to_process:
        item["temp_hdbet"].parent.mkdir(parents=True, exist_ok=True)
        if not item["mask_file"].exists():
            paths_for_hdbet_in.append(str(item["raw_path"]))
            paths_for_hdbet_out.append(str(item["temp_hdbet"]))
            
    if paths_for_hdbet_in:
        print(f"--- Batch HD-BET Skull Masking ({len(paths_for_hdbet_in)} files) ---")
        try:
            run_hd_bet(mri_fnames=paths_for_hdbet_in, output_fnames=paths_for_hdbet_out, 
                       mode="fast", device=0 if use_gpu else "cpu", do_tta=False)
        except Exception as e:
            print(f"Error during HD-BET: {e}")

    # --- Registration and Z-Score Normalisation ---
    print("--- Registration and Z-Score normalisation ---")
    
    fixed_template = ants.image_read(str(template_path))
    success_count = 0
    
    for item in tqdm(files_to_process, desc="Processing"):
        if item["png_out_dir"].exists() or not item["mask_file"].exists(): 
            continue
        
        success = process_single_scan(
            raw_path=item["raw_path"], 
            mask_file=item["mask_file"], 
            png_out_dir=item["png_out_dir"],
            stem=item["stem"],
            fixed_template=fixed_template
        )
        
        if success:
            success_count += 1
            eortc_src = item["raw_path"].parent / "eortc_scores.csv"
            eortc_dst = output_dir / item["raw_path"].relative_to(input_dir).parent / "eortc_scores.csv"
            if eortc_src.exists() and not eortc_dst.exists():
                shutil.copy2(eortc_src, eortc_dst)
            
        # Manually trigger python garbage collection to clean up lingering C++ ITK hooks
        gc.collect()

    print("Cleanup ---")
    if success_count > 0 and temp_dir.exists():
        shutil.rmtree(temp_dir)
        
    print(f"Pipeline Complete. Successfully processed {success_count} files")

In [ ]:
master_brats_pipeline(
    input_dir="NII_DIR",  # Replace with actual path to raw NII files
    output_dir="OUTPUT_DIR",  # Replace with actual path to output directory
    template_path="TEMPLATE_PATH",  # Replace with actual path to template file
    use_gpu=True
)